In [18]:
import requests
from bs4 import BeautifulSoup
from time import sleep
from urllib.parse import unquote

import pandas as pd
import numpy as np
import re

In [2]:
HEADERS = {"User-Agent": "qangos-scraper/1.0(contact:q@bradtaba.org)"}
API = "https://onepiece.fandom.com/api.php"

character_columns = [
    'Official English Name', 
    'Debut', 
    'Affiliations', 
    'Occupations', 
    'Origin', 
    'Status', 
    'Age', 
    'Birthday', 
    'Height', 
    'Blood Type', 
    'Bounty'
]

In [4]:
def fetch_onepiece(page, sleep_time=2):
    params = {
        "action": "parse",
        "page": unquote(page),
        "prop": "text",
        "format": "json"
    }

    try:
        r = requests.get(API, params=params, headers=HEADERS, timeout=30)
        data = r.json()
        html = data["parse"]["text"]["*"]
        sleep(sleep_time)
    except:
        return False
    else:
        print(f"Page has been successfully fetched! {page}")

    return html


In [5]:
character_page = fetch_onepiece("List_of_Canon_Characters")
soup = BeautifulSoup(character_page, 'html5lib')

char_table = []
for i in soup.tr.find_next_siblings():
    char_data = {
        'name': i.a.text,
        'link': i.a['href'][2:].split('/')[-1]
    }
    char_table.append(char_data)
    
char_table = pd.DataFrame(char_table)

Page has been successfully fetched! List_of_Canon_Characters


In [73]:
status_pattern = r'\(.*?$'
status_regex = re.compile(status_pattern)

name_pattern = r' \(.*?\)'
name_regex = re.compile(name_pattern)

bday_pattern = r'\b[a-z]+\b \d{1,2}'
bday_regex = re.compile(bday_pattern, re.I)

age_pattern = r'\d+'
age_regex = re.compile(age_pattern)

height_pattern = r'\d+\s*cm'
height_regex = re.compile(height_pattern)

height_m_pattern = r'(\d+)\s*m'
height_m_regex = re.compile(height_m_pattern)

hyperlink_pattern = r'\[\d+\]'
hyperlink_regex = re.compile(hyperlink_pattern)

debut_pattern = r'Chapter (\d+)'
debut_regex = re.compile(debut_pattern, re.I)

bounty_edgecase_pattern = r'\(At least (.*?)\)'
bounty_edgecase_regex = re.compile(bounty_edgecase_pattern, re.I)

bounty_edgecase2_pattern = r'\(([0-9,]*)\)'
bounty_edgecase2_regex = re.compile(bounty_edgecase2_pattern, re.I)

In [7]:
armament_page = fetch_onepiece('Haki/Armament_Haki')
armament_soup = BeautifulSoup(armament_page, 'html5lib')

armament_users = []

for i in armament_soup.find_all('small'):
    armament_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Armament_Haki


In [8]:
conq_page = fetch_onepiece('Haki/Supreme_King_Haki')
conq_soup = BeautifulSoup(conq_page, 'html5lib')

conq_users = []

for i in conq_soup.find_all('small'):
    conq_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Supreme_King_Haki


In [9]:
observation_page = fetch_onepiece('Haki/Observation_Haki')
observation_soup = BeautifulSoup(observation_page, 'html5lib')

observation_users = []

for i in observation_soup.find_all('small'):
    observation_users.append(status_regex.sub('',i.text).strip())

Page has been successfully fetched! Haki/Observation_Haki


In [74]:
def clean_data(luffy_dict):
    if 'Official English Name' in luffy_dict: 
        luffy_dict['Official English Name'] = name_regex.sub('', luffy_dict['Official English Name'].split(';')[0])

    if 'Affiliations' in luffy_dict:
        luffy_dict['Affiliations'] = luffy_dict['Affiliations'].split(';')[0]

    if 'Occupations' in luffy_dict:
        luffy_dict['Occupations'] = luffy_dict['Occupations'].split(';')[0]

    if 'Bounty' in luffy_dict:
        if luffy_dict['Bounty'].lower().startswith('(') and luffy_dict['Bounty'].lower().endswith(')'):
            luffy_dict['Bounty'] = luffy_dict['Bounty'][1:-1]

        if isinstance(luffy_dict['Bounty'], int):
            luffy_dict['Bounty'] = luffy_dict['Bounty']
        elif bounty_edgecase_regex.findall(luffy_dict['Bounty']):
            luffy_dict['Bounty'] = int(''.join(bounty_edgecase_regex.findall(luffy_dict['Bounty'])[-1].split(',')))
        elif (luffy_dict['Bounty'].lower().startswith('at')):
            if bounty_edgecase2_regex.findall(luffy_dict['Bounty']):
                luffy_dict['Bounty'] = int(''.join(bounty_edgecase2_regex.findall(luffy_dict['Bounty'])[-1].split(',')))
            else:
                luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[-1]).split(',')))
        elif (luffy_dict['Bounty'].lower().startswith('unknown')):
            luffy_dict['Bounty'] = 0
        elif luffy_dict['Bounty'].lower().startswith('¥'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'][1:].split(' ')[0]).split(',')))
        elif luffy_dict['Bounty'].lower().startswith('over'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[1]).split(',')))
        elif luffy_dict['Bounty'].lower().startswith('less'):
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[2]).split(',')))
        elif bounty_edgecase2_regex.findall(luffy_dict['Bounty']):
            luffy_dict['Bounty'] = int(''.join(bounty_edgecase2_regex.findall(luffy_dict['Bounty'])[-1].split(',')))
        else:
            luffy_dict['Bounty'] = int(''.join((luffy_dict['Bounty'].split(' ')[0]).split(',')))

    if 'Origin' in luffy_dict:
        luffy_dict['Origin'] = ' '.join(luffy_dict['Origin'].split()[:2])

    if 'Birthday' in luffy_dict:
        luffy_dict['Birthday'] = bday_regex.findall(luffy_dict['Birthday'])[0]

    if 'Age' in luffy_dict:
        ages = age_regex.findall(luffy_dict['Age'].split(';')[-1].strip())
        if ages:
            luffy_dict['Age'] = int(ages[0])
        elif luffy_dict['Age'] == 'Multiple Centuries':
            luffy_dict['Age'] = 100
        else:
            luffy_dict['Age'] = 0

    if 'Height' in luffy_dict:
        cm = height_regex.findall(luffy_dict['Height'])
        if cm:
            luffy_dict['Height'] = int(cm[-1][:-3])
        elif height_m_regex.findall(luffy_dict['Height']):
            luffy_dict['Height'] = int(height_m_regex.findall(luffy_dict['Height'])[-1])*100
        else:
            luffy_dict['Height'] = np.nan

    if 'Debut' in luffy_dict:
        chap = debut_regex.findall(luffy_dict['Debut'])
        if chap:
            luffy_dict['Debut'] = int(chap[0])
        else:
            luffy_dict['Debut'] = np.nan

    return luffy_dict

In [11]:
character_info_list = []

In [77]:
for link in char_table['link'].to_list()[1239:]:
    luffy_page = fetch_onepiece(link)
    if not luffy_page:
        print(f'skip: {link}. index {int(char_table[char_table['link']==link].index[0])}')
        print('Next:', int(char_table[char_table['link']==link].index[0])+1)
        continue
    luffy_soup = BeautifulSoup(luffy_page, 'html5lib')

    luffy_dict = {}
    for i in luffy_soup.find_all(class_='pi-data-label pi-secondary-font'):
        if i.text[:-1] in character_columns:
            luffy_dict[i.text[:-1]] = hyperlink_regex.sub(' ', i.find_next_sibling().text).strip()

    fruit_data = luffy_soup.find_all('span', class_='pi-data-label pi-secondary-font')
    if fruit_data:
        luffy_dict['Devil Fruit'] = 1
        luffy_dict['Fruit Type'] = hyperlink_regex.sub(' ', fruit_data[-1].find_next_sibling().text).strip()
    else:
        luffy_dict['Devil Fruit'] = 0

    if 'Official English Name' in luffy_dict:
        luffy_dict['Observation Haki'] = 1 if luffy_dict['Official English Name'] in observation_users else 0
        luffy_dict['Armament Haki'] = 1 if luffy_dict['Official English Name'] in armament_users else 0
        luffy_dict['Conquerers Haki'] = 1 if luffy_dict['Official English Name'] in conq_users else 0
    else:
        luffy_dict['Observation Haki'] = 0
        luffy_dict['Armament Haki'] = 0
        luffy_dict['Conquerers Haki'] = 0

    character_info_list.append(clean_data(luffy_dict))
    print('Next:', int(char_table[char_table['link']==link].index[0])+1)

Page has been successfully fetched! Sanji
Next: 1240
Page has been successfully fetched! Sanjuan_Wolf
Next: 1241
Page has been successfully fetched! Medaka_Mermaid_Quintuplets
Next: 627
Page has been successfully fetched! Sapi
Next: 1243
Page has been successfully fetched! Sarahebi
Next: 1244
Page has been successfully fetched! Sarai
Next: 1245
Page has been successfully fetched! Sarfunkel
Next: 1246
Page has been successfully fetched! Sarie_Nantokanette
Next: 1247
Page has been successfully fetched! Sarquiss
Next: 1248
Page has been successfully fetched! Saru
Next: 1249
Page has been successfully fetched! Sarutobi
Next: 1250
Page has been successfully fetched! Sasaki
Next: 1251
Page has been successfully fetched! Satchels_Maffey
Next: 1252
Page has been successfully fetched! Satori
Next: 1253
Page has been successfully fetched! Sauce
Next: 1254
Page has been successfully fetched! Scales
Next: 1255
Page has been successfully fetched! Scarlett
Next: 1256
Page has been successfully fetch

In [63]:
luffy_dict

{'Official English Name': 'Sanji',
 'Debut': 'Chapter 43; Episode 20',
 'Affiliations': 'Straw Hat Pirates',
 'Occupations': 'Cook',
 'Origin': 'North Blue',
 'Status': 'Alive',
 'Age': '19 (debut) 21 (after timeskip)',
 'Birthday': 'March 2nd',
 'Height': '177 cm (5\'10") (debut) 180 cm (5\'11") (after timeskip)',
 'Blood Type': 'S (RH-)',
 'Bounty': '1,032,000,000 330,000,000 177,000,000 (Only Alive) 77,000,000',
 'Devil Fruit': 0,
 'Observation Haki': 1,
 'Armament Haki': 1,
 'Conquerers Haki': 0}

In [47]:
height_m_regex.findall(luffy_dict['Height'])[-1]

'715 m'

In [79]:
character_info = pd.DataFrame(character_info_list)
character_info.tail()

,Official English Name,Debut,Affiliations,Occupations,Status,Birthday,Devil Fruit,Observation Haki,Armament Haki,Conquerers Haki,Origin,Age,Height,Blood Type,Fruit Type,Bounty
1548,Zodia,553.0,Whitebeard Pirates,Pirate,Alive,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1549,Burr,533.0,Marines G-1 Branch,Marine Lieutenant,Alive,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,1000000.0
1550,Zukka,564.0,Ally of the Whitebeard Pirates,Pirate,Alive,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1551,Zunesha,802.0,NaN,NaN,Alive,December 18,0,0,0,0,Grand Line,1000.0,100.0,NaN,NaN,NaN
1552,Zururin,621.0,NaN,NaN,Alive,NaN,0,0,0,0,Grand Line,NaN,NaN,NaN,NaN,NaN


In [81]:
character_info.to_csv('character_info_v1.csv')

In [80]:
char_table['link'].to_list()[1554]

IndexError: list index out of range

In [ ]:
luffy_page = fetch_onepiece(char_table['link'].to_list()[1554])
if not luffy_page:
    print(f'skip: {link}. index {int(char_table[char_table['link']==link].index[0])}')
    print('Next:', int(char_table[char_table['link']==link].index[0])+1)
    continue
luffy_soup = BeautifulSoup(luffy_page, 'html5lib')

luffy_dict = {}
for i in luffy_soup.find_all(class_='pi-data-label pi-secondary-font'):
    if i.text[:-1] in character_columns:
        luffy_dict[i.text[:-1]] = hyperlink_regex.sub(' ', i.find_next_sibling().text).strip()

fruit_data = luffy_soup.find_all('span', class_='pi-data-label pi-secondary-font')
if fruit_data:
    luffy_dict['Devil Fruit'] = 1
    luffy_dict['Fruit Type'] = hyperlink_regex.sub(' ', fruit_data[-1].find_next_sibling().text).strip()
else:
    luffy_dict['Devil Fruit'] = 0

if 'Official English Name' in luffy_dict:
    luffy_dict['Observation Haki'] = 1 if luffy_dict['Official English Name'] in observation_users else 0
    luffy_dict['Armament Haki'] = 1 if luffy_dict['Official English Name'] in armament_users else 0
    luffy_dict['Conquerers Haki'] = 1 if luffy_dict['Official English Name'] in conq_users else 0
else:
    luffy_dict['Observation Haki'] = 0
    luffy_dict['Armament Haki'] = 0
    luffy_dict['Conquerers Haki'] = 0

character_info_list.append(clean_data(luffy_dict))
print('Next:', int(char_table[char_table['link']==link].index[0])+1)